In [0]:
SELECT COUNT(*) FROM workspace.default.orders;

COUNT(*)
1103


In [0]:
DESCRIBE workspace.default.order_items;

col_name,data_type,comment
item_id,bigint,null
order_id,bigint,null
product_id,bigint,null
quantity,bigint,null
unit_price,double,null


Revenue by category:

In [0]:
SELECT p.category, ROUND(SUM(oi.quantity * oi.unit_price), 0) AS revenue
FROM workspace.default.order_items oi
JOIN workspace.default.orders   o ON o.order_id   = oi.order_id
JOIN workspace.default.products p ON p.product_id = oi.product_id
WHERE o.status = 'Delivered'
GROUP BY p.category
ORDER BY revenue DESC;

category,revenue
Electronics,830913.0
Fashion,653328.0
Home,521969.0
Sports,455736.0
Books,307538.0


Monthly revenue with growth:

In [0]:
WITH monthly AS (
  SELECT date_format(CAST(o.order_date AS DATE), 'yyyy-MM') AS month,
         ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
  FROM workspace.default.orders o
  JOIN workspace.default.order_items oi ON oi.order_id = o.order_id
  WHERE o.status = 'Delivered'
  GROUP BY 1
)
SELECT month, revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_month,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1) AS growth_pct
FROM monthly
ORDER BY month;

month,revenue,prev_month,growth_pct
2025-01,4745.0,null,null
2025-02,32070.0,4745.0,575.9
2025-03,54663.0,32070.0,70.4
2025-04,116526.0,54663.0,113.2
2025-05,129721.0,116526.0,11.3
2025-06,136299.0,129721.0,5.1
2025-07,216852.0,136299.0,59.1
2025-08,313183.0,216852.0,44.4
2025-09,241846.0,313183.0,-22.8
2025-10,475201.0,241846.0,96.5


Top 3 products per category:


In [0]:
WITH product_sales AS (
  SELECT p.category, p.product_name,
         SUM(oi.quantity * oi.unit_price) AS revenue
  FROM workspace.default.order_items oi
  JOIN workspace.default.orders   o ON o.order_id   = oi.order_id
  JOIN workspace.default.products p ON p.product_id = oi.product_id
  WHERE o.status = 'Delivered'
  GROUP BY p.category, p.product_name
),
ranked AS (
  SELECT *, RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS rnk
  FROM product_sales
)
SELECT category, rnk, product_name, ROUND(revenue, 0) AS revenue
FROM ranked WHERE rnk <= 3
ORDER BY category, rnk;

category,rnk,product_name,revenue
Books,1,Clean Code,62930.0
Books,2,Data Science Handbook,48739.0
Books,3,Deep Learning Guide,42957.0
Electronics,1,Mechanical Keyboard,181445.0
Electronics,2,Smart Watch,160954.0
Electronics,3,Webcam,159936.0
Fashion,1,Running Shoes,155948.0
Fashion,2,Denim Jacket,124950.0
Fashion,3,Sneakers,93564.0
Home,1,Air Fryer,264947.0


Repeat vs one-time customers:

In [0]:
WITH orders_per_customer AS (
  SELECT customer_id, COUNT(*) AS n_orders
  FROM workspace.default.orders
  WHERE status = 'Delivered'
  GROUP BY customer_id
)
SELECT CASE WHEN n_orders = 1 THEN 'One-time' ELSE 'Repeat (2+)' END AS customer_type,
       COUNT(*) AS customers
FROM orders_per_customer
GROUP BY 1;

customer_type,customers
Repeat (2+),251
One-time,139
